# 1) Modelo de projeção da câmera Pinhole:

Uso principal: renderização 2D, reconstrução, segmentação

<b>screen space $\xleftarrow{\div w}$ screen space homogeneous $\xleftarrow{K}$ view space  $\xleftarrow{[R|t]}$ object space </b>

$
w 
\begin{bmatrix} 
u \\
v \\
1 
\end{bmatrix}= 
\underbrace{ 
        \underbrace{ 
            \begin{bmatrix} 
                f_x & 0 & c_x & 0 \\
                0 & f_y & c_y & 0 \\
                0 & 0 & 1 & 0 
            \end{bmatrix} 
        }_{Intrinsic \ parameters \ [K_{3\times 3}|0]} 
        \underbrace{ 
            \begin{bmatrix} 
                R_{3 \times 3} & t_{3\times 1} \\
                0_{1 \times 3} & 1  
            \end{bmatrix} 
        }_{Extrinsic \ parameters \ T_{4\times 4}} 
}_{P_{3\times 4}} 
\begin{bmatrix} 
X \\
Y \\
Z\\
1 
\end{bmatrix}
$

- A entrada $\mathbf{X} = (X, Y, Z, 1)$ é coordenada de um ponto no espaço-objeto (<i>world space</i>);
- A saída $\mathbf{u} = (u, v) $ é coordenada $(col, lin)$ em pixels no espaço-imagem (<i>screen space</i>)
- $[R|t]$ é a notação abreviada, sem a 4ª linha, da matriz extrínseca $T$, transforma pontos do espaço-objeto para o espaço-câmera (<i>view space</i>);
    - A inversa da matriz extrínseca é a matriz de posição da câmera: $T^{-1}= $ <i>camera pose</i> $= \begin{bmatrix}R^T & - R^TC\\0&1\end{bmatrix}$
- $K$ é a matriz intrínseca, transforma pontos do espaço-câmera para o espaço-imagem (<i>screen space</i>);
- $P$ é a matriz de projeção;

# 2) Modelo de projeção alinhado com APIs de gráficos (OpenGL/DirectX):

Uso principal: pipeline gráfico, clipping, depth test, rasterização

<b>screen space $\xleftarrow{viewport \ transform}$ Normalized Device Coordinates (NDC) [-1,1] $\xleftarrow{ \div w}$ Clip Space $\xleftarrow{P}$ view space $\xleftarrow{[R|t]}$ object space</b>

$w
\begin{bmatrix} 
x_{ndc} \\
y_{ndc} \\
z_{ndc} \\
1
\end{bmatrix} = 
\begin{bmatrix} 
x_{clip} \\
y_{clip} \\
z_{clip} \\
w
\end{bmatrix} = 
\underbrace{ 
   \begin{bmatrix} 
         2n/(d-e) & 0 & (d+e)/(d-e) & 0 \\
         0 & 2n/(c-b) & (c+b)/(c-b) & 0 \\
         0 & 0 & -(f+n)/(f-n) & -2fn/(f-n) \\
         0 & 0 & -1 & 0 
   \end{bmatrix} 
}_{Projection \ matrix \ P_{4\times4}} 
\underbrace{ 
   \begin{bmatrix} 
         R_{3 \times 3} & t_{3\times 1} \\
         0_{1 \times 3} & 1  
   \end{bmatrix} 
}_{View \ matrix} 
\begin{bmatrix} 
X \\
Y \\
Z\\
1 
\end{bmatrix}
$

- A entrada $\mathbf{X} = (X, Y, Z, 1)$ é coordenada de um ponto no espaço-objeto (<i>world space</i>);
- A saída $\mathbf{x_{clip}} = (\tilde{x}, \tilde{y}, \tilde{z}, w) $ é coordenada do ponto no espaço-recorte (<i>clip space</i>);
- $[R|t]$ é a notação abreviada, sem a 4ª linha, da view matrix (= $T$), transforma pontos do espaço-objeto para o espaço-câmera (<i>view space</i>);
- $n,f \rightarrow$ near e far plane;
- $e,d,c,b \rightarrow$ frustum bounds (esquerda, direita, cima, baixo);
- $P$ é a matriz de projeção neste caso, transforma do espaço-câmera para o espaço de recorte (<i>clip space</i>);
- A operação <i>perspective divide</i> transforma pontos do espaço-recorte para as Coordenadas Normalizadas do Dispositivo (NDC):
   - $\mathbf{x_{ndc}} = (x_{ndc},y_{ndc},z_{ndc},1) = \mathbf{x_{clip}} \div w$
   - $x_{ndc} \in [-1,1]$;
   - $y_{ndc} \in [-1,1]$;
   - $z_{ndc} \in [0,1]$;
- Dadas a largura e altura da imagem ($W$ e $H$), a operação <i>viewport mapping</i> transforma pontos do NDC para o espaço-imagem (<i>screen space</i>);
    - $u = \frac{x_{ndc} + 1}{2} \cdot W$
    - $v = \frac{1-y_{ndc}}{2} \cdot H$

A matriz de projeção é implementada da seguinte forma:

```python
"""
Taken from torch-splatting/gaussian_splatting/utils/camera_utils.py
Also  from gaussian-splatting/utils/graphics_utils.py
"""

def getProjectionMatrix(znear, zfar, fovX, fovY):
    tanHalfFovY = math.tan((fovY / 2))
    tanHalfFovX = math.tan((fovX / 2))

    top = tanHalfFovY * znear
    bottom = -top
    right = tanHalfFovX * znear
    left = -right

    P = torch.zeros(4, 4)

    z_sign = 1.0
    P[0, 0] = 2.0 * znear / (right - left)
    P[1, 1] = 2.0 * znear / (top - bottom)
    P[0, 2] = (right + left) / (right - left)
    P[1, 2] = (top + bottom) / (top - bottom)
    P[3, 2] = z_sign
    P[2, 2] = z_sign * zfar / (zfar - znear)
    P[2, 3] = -(zfar * znear) / (zfar - znear)
    return P

def fov2focal(fov, pixels):
    return pixels / (2 * math.tan(fov / 2))

def focal2fov(focal, pixels):
    return 2*math.atan(pixels/(2*focal))
```


- Chamada com zfar = 100.0  e znear = 0.01

# 3) Frustum culling (recorte de tronco):

### Vanilla 3DGS:

```cpp

"""
Taken from diff-gaussian-rasterization/cuda_rasterizer/auxiliary.h

Conceptually wrong because in gaussian_renderer/__init__.py
    raster_settings = GaussianRasterizationSettings(
        ...
        viewmatrix=viewpoint_camera.world_view_transform,
        projmatrix=viewpoint_camera.full_proj_transform,
        ...
    )

And in scene/cameras.py
    self.world_view_transform = torch.tensor(getWorld2View2(R, T, trans, scale)).transpose(0, 1).cuda()
    self.projection_matrix = getProjectionMatrix(znear=self.znear, zfar=self.zfar, fovX=self.FoVx, fovY=self.FoVy).transpose(0,1).cuda()
    self.full_proj_transform = (self.world_view_transform.unsqueeze(0).bmm(self.projection_matrix.unsqueeze(0))).squeeze(0)

So, projmatrix here = full_proj_transform = viewmatrix.T @ projmatrix.T, 
"""

__forceinline__ __device__ bool in_frustum(int idx,
	const float* orig_points,
	const float* viewmatrix,
	const float* projmatrix,
	bool prefiltered,
	float3& p_view)
{
	float3 p_orig = { orig_points[3 * idx], orig_points[3 * idx + 1], orig_points[3 * idx + 2] };

	// Bring points to screen space
	float4 p_hom = {projmatrix[0] * p_orig.x + projmatrix[4] * p_orig.y + projmatrix[8] * p_orig.z + projmatrix[12],
			projmatrix[1] * p_orig.x + projmatrix[5] * p_orig.y + projmatrix[9] * p_orig.z + projmatrix[13],
			projmatrix[2] * p_orig.x + projmatrix[6] * p_orig.y + projmatrix[10] * p_orig.z + projmatrix[14],
			projmatrix[3] * p_orig.x + projmatrix[7] * p_orig.y + projmatrix[11] * p_orig.z + projmatrix[15]
			};
	float p_w = 1.0f / (p_hom.w + 0.0000001f);
	float3 p_proj = { p_hom.x * p_w, p_hom.y * p_w, p_hom.z * p_w };
	p_view = {viewmatrix[0] * p_orig.x + viewmatrix[4] * p_orig.y + viewmatrix[8] * p_orig.z + viewmatrix[12],
		viewmatrix[1] * p_orig.x + viewmatrix[5] * p_orig.y + viewmatrix[9] * p_orig.z + viewmatrix[13],
		viewmatrix[2] * p_orig.x + viewmatrix[6] * p_orig.y + viewmatrix[10] * p_orig.z + viewmatrix[14],
		};

	if (p_view.z <= 0.2f)// || ((p_proj.x < -1.3 || p_proj.x > 1.3 || p_proj.y < -1.3 || p_proj.y > 1.3)))
	{
		if (prefiltered)
		{
			printf("Point is filtered although prefiltered is set. This shouldn't happen!");
			__trap();
		}
		return false;
	}
	return true;
}
```

- p_proj está no espaço NDC, ou seja, entre $[-1, 1]$;
- p_view está no <i>view space</i>;
- Verifica se um ponto 3D está na frente da câmera, ou seja, um ponto no espaço-câmera possui $z > 0.2$;
- A linha comentada (p_proj.x/y < -1.3 ou > 1.3) faria o teste completo do frustum NDC, mas está desativada.
- A fraqueza dessa seleção é reconhecida no paper que introduziu 3DGS:

    ```txt
    "One reason for these popping artifacts is the trivial rejection 
    of Gaussians via a guard band in the rasterizer.
    A more principled culling approach would alleviate these artifacts." (KERBL et al., 2023)
    ```

### Torch-Splatting:

```python

"""
Taken from torch-splatting/gaussian_splatting/gauss_render.py
"""

def build_covariance_2d(
    mean3d, cov3d, viewmatrix, fov_x, fov_y, focal_x, focal_y
):
    # The following models the steps outlined by equations 29
	# and 31 in "EWA Splatting" (Zwicker et al., 2002). 
	# Additionally considers aspect / scaling of viewport.
	# Transposes used to account for row-/column-major conventions.
    tan_fovx = math.tan(fov_x * 0.5)
    tan_fovy = math.tan(fov_y * 0.5)
    t = (mean3d @ viewmatrix[:3,:3]) + viewmatrix[-1:,:3]

    # truncate the influences of gaussians far outside the frustum.
    tx = (t[..., 0] / t[..., 2]).clip(min=-tan_fovx*1.3, max=tan_fovx*1.3) * t[..., 2]
    ty = (t[..., 1] / t[..., 2]).clip(min=-tan_fovy*1.3, max=tan_fovy*1.3) * t[..., 2]
    tz = t[..., 2]

    # Eq.29 locally affine transform 
    # perspective transform is not affine so we approximate with first-order taylor expansion
    # notice that we multiply by the intrinsic so that the variance is at the sceen space
    J = torch.zeros(mean3d.shape[0], 3, 3).to(mean3d)
    J[..., 0, 0] = 1 / tz * focal_x
    J[..., 0, 2] = -tx / (tz * tz) * focal_x
    J[..., 1, 1] = 1 / tz * focal_y
    J[..., 1, 2] = -ty / (tz * tz) * focal_y
    # J[..., 2, 0] = tx / t.norm(dim=-1) # discard
    # J[..., 2, 1] = ty / t.norm(dim=-1) # discard
    # J[..., 2, 2] = tz / t.norm(dim=-1) # discard
    W = viewmatrix[:3,:3].T # transpose to correct viewmatrix
    cov2d = J @ W @ cov3d @ W.T @ J.permute(0,2,1)
    
    # add low pass filter here according to E.q. 32
    filter = torch.eye(2,2).to(cov2d) * 0.3
    return cov2d[:, :2, :2] + filter[None]
```

- t são os pontos mean3d projetados para o espaço-câmera (<i>view space</i>);
- <span style="color:orange">tan_fovx</span> e <span style="color:green">tan_fovx</span> são na verdade, <span style="color:orange">tan_half_fovX_camera</span> e <span style="color:green">tan_half_fovY_camera</span>;
- t[0]/t[2] = <span style="color:orange">tan_half_fovX_camera</span> de um ponto do espaço-objeto transformado para o espaço desta câmera (<i>view space</i>)
    - Se o valor dessa tangente for muito maior que <span style="color:orange">tan_fovx</span> significa que o ponto está muito a direita da imagem;
    - Se o valor dessa tangente for muito menor que <span style="color:orange">-tan_fovx</span> significa que o ponto está muito a esquerda da imagem;
    - Um limite de 30% para mais e para menos para t[0]/t[2] é estabelecido heuristicamente;
- Da mesma maneira, t[1]/t[2] = <span style="color:green">tan_half_fovY_camera</span> de um ponto do espaço-objeto transformado para o espaço desta câmera (<i>view space</i>)
    - Se o valor dessa tangente for muito maior que <span style="color:green">tan_fovy</span> significa que o ponto está muito acima da imagem;
    - Se o valor dessa tangente for muito menor que <span style="color:green">-tan_fovy</span> significa que o ponto está muito abaixo da imagem;
    - Um limite de 30% para mais e para menos para t[1]/t[2] é estabelecido heuristicamente;


# 4) Sobre uma projeção adequada para os pontos 3D:

#### Get Colmap Data to work on:

In [2]:
## Imports
 
import os
import numpy as np
import collections

## Functions and classes
"""
    Taken from https://github.com/graphdeco-inria/gaussian-splatting/blob/main/scene/colmap_loader.py
""" 

CameraModel = collections.namedtuple(
    "CameraModel", ["model_id", "model_name", "num_params"])
Camera = collections.namedtuple(
    "Camera", ["id", "model", "width", "height", "params"])
BaseImage = collections.namedtuple(
    "Image", ["id", "qvec", "tvec", "camera_id", "name", "xys", "point3D_ids"])
Point3D = collections.namedtuple(
    "Point3D", ["id", "xyz", "rgb", "error", "image_ids", "point2D_idxs"])
CAMERA_MODELS = {
    CameraModel(model_id=0, model_name="SIMPLE_PINHOLE", num_params=3),
    CameraModel(model_id=1, model_name="PINHOLE", num_params=4),
    CameraModel(model_id=2, model_name="SIMPLE_RADIAL", num_params=4),
    CameraModel(model_id=3, model_name="RADIAL", num_params=5),
    CameraModel(model_id=4, model_name="OPENCV", num_params=8),
    CameraModel(model_id=5, model_name="OPENCV_FISHEYE", num_params=8),
    CameraModel(model_id=6, model_name="FULL_OPENCV", num_params=12),
    CameraModel(model_id=7, model_name="FOV", num_params=5),
    CameraModel(model_id=8, model_name="SIMPLE_RADIAL_FISHEYE", num_params=4),
    CameraModel(model_id=9, model_name="RADIAL_FISHEYE", num_params=5),
    CameraModel(model_id=10, model_name="THIN_PRISM_FISHEYE", num_params=12)
}
CAMERA_MODEL_IDS = dict([(camera_model.model_id, camera_model)
                         for camera_model in CAMERA_MODELS])
CAMERA_MODEL_NAMES = dict([(camera_model.model_name, camera_model)
                           for camera_model in CAMERA_MODELS])

def qvec2rotmat(qvec):
    return np.array([
        [1 - 2 * qvec[2]**2 - 2 * qvec[3]**2,
         2 * qvec[1] * qvec[2] - 2 * qvec[0] * qvec[3],
         2 * qvec[3] * qvec[1] + 2 * qvec[0] * qvec[2]],
        [2 * qvec[1] * qvec[2] + 2 * qvec[0] * qvec[3],
         1 - 2 * qvec[1]**2 - 2 * qvec[3]**2,
         2 * qvec[2] * qvec[3] - 2 * qvec[0] * qvec[1]],
        [2 * qvec[3] * qvec[1] - 2 * qvec[0] * qvec[2],
         2 * qvec[2] * qvec[3] + 2 * qvec[0] * qvec[1],
         1 - 2 * qvec[1]**2 - 2 * qvec[2]**2]])

def rotmat2qvec(R):
    Rxx, Ryx, Rzx, Rxy, Ryy, Rzy, Rxz, Ryz, Rzz = R.flat
    K = np.array([
        [Rxx - Ryy - Rzz, 0, 0, 0],
        [Ryx + Rxy, Ryy - Rxx - Rzz, 0, 0],
        [Rzx + Rxz, Rzy + Ryz, Rzz - Rxx - Ryy, 0],
        [Ryz - Rzy, Rzx - Rxz, Rxy - Ryx, Rxx + Ryy + Rzz]]) / 3.0
    eigvals, eigvecs = np.linalg.eigh(K)
    qvec = eigvecs[[3, 0, 1, 2], np.argmax(eigvals)]
    if qvec[0] < 0:
        qvec *= -1
    return qvec

class Image(BaseImage):
    def qvec2rotmat(self):
        return qvec2rotmat(self.qvec)
    
def read_intrinsics_text(path):
    """
    Taken from https://github.com/colmap/colmap/blob/dev/scripts/python/read_write_model.py
    """
    cameras = {}
    with open(path, "r") as fid:
        while True:
            line = fid.readline()
            if not line:
                break
            line = line.strip()
            if len(line) > 0 and line[0] != "#":
                elems = line.split()
                camera_id = int(elems[0])
                model = elems[1]
                assert model == "PINHOLE", "While the loader support other types, the rest of the code assumes PINHOLE"
                width = int(elems[2])
                height = int(elems[3])
                params = np.array(tuple(map(float, elems[4:])))
                cameras[camera_id] = Camera(id=camera_id, model=model,
                                            width=width, height=height,
                                            params=params)
    return cameras

def intr_from_pinhole_camera(camera):
    if camera.model=='SIMPLE_PINHOLE':
        # https://github.com/colmap/colmap/blob/main/src/colmap/sensor/models.h#L250-L256
        f, cx, cy =  camera.params
        fx = fy = f
    elif camera.model=='PINHOLE':
        # https://github.com/colmap/colmap/blob/main/src/colmap/sensor/models.h#L265-L271
        fx, fy, cx, cy =  camera.params
    return np.array([[fx, 0, camera.width/2],
                     [0, fy, camera.height/2],
                     [0, 0, 1]])

def read_extrinsics_text(path):
    """
    Taken from https://github.com/colmap/colmap/blob/dev/scripts/python/read_write_model.py
    """
    images = {}
    with open(path, "r") as fid:
        while True:
            line = fid.readline()
            if not line:
                break
            line = line.strip()
            if len(line) > 0 and line[0] != "#":
                elems = line.split()
                image_id = int(elems[0])
                qvec = np.array(tuple(map(float, elems[1:5])))
                tvec = np.array(tuple(map(float, elems[5:8])))
                camera_id = int(elems[8])
                image_name = elems[9]
                elems = fid.readline().split()
                xys = np.column_stack([tuple(map(float, elems[0::3])),
                                       tuple(map(float, elems[1::3]))])
                point3D_ids = np.array(tuple(map(int, elems[2::3])))
                images[image_id] = Image(
                    id=image_id, qvec=qvec, tvec=tvec,
                    camera_id=camera_id, name=image_name,
                    xys=xys, point3D_ids=point3D_ids)
    return images

def extr_from_image(image):
    R = image.qvec2rotmat()
    t = np.array(image.tvec)
    Rt = np.eye(4)
    Rt[:3, :3] = R
    Rt[:3, 3] = t
    return Rt

def read_points3D_text(path):
    """
    see: src/base/reconstruction.cc
        void Reconstruction::ReadPoints3DText(const std::string& path)
        void Reconstruction::WritePoints3DText(const std::string& path)
    """
    num_points = 0
    with open(path, "r") as fid:
        while True:
            line = fid.readline()
            if not line:
                break
            line = line.strip()
            if len(line) > 0 and line[0] != "#":
                num_points += 1

    points3d = {}
    with open(path, "r") as fid:
        while True:
            line = fid.readline()
            if not line:
                break
            line = line.strip()
            if len(line) > 0 and line[0] != "#":
                elems = line.split()
                id = int(elems[0])
                xyz = np.array(tuple(map(float, elems[1:4])))
                rgb = np.array(tuple(map(int, elems[4:7])))
                error = np.array(float(elems[7]))
                track = np.array(elems[8:]) # as (image_id, point2D_idx)
                image_ids = track[0::2].astype(int)
                point2D_idxs = track[1::2].astype(int)
                points3d[id]= Point3D(id = id,
                                       xyz = xyz,
                                       rgb = rgb,
                                       error = error,
                                       image_ids = image_ids,
                                       point2D_idxs = point2D_idxs)
    return points3d

## Get Colmap sparse model
folder = "../aparecida/undistorted_8"

cameras_intrinsic_file = os.path.join(folder, "sparse", "cameras.txt")
cam_intrinsics = read_intrinsics_text(cameras_intrinsic_file)

cameras_extrinsic_file = os.path.join(folder, "sparse", "images.txt")
cam_extrinsics = read_extrinsics_text(cameras_extrinsic_file)

points3D_file = os.path.join(folder, "sparse", "points3D.txt")
points3d = read_points3D_text(points3D_file)

### Uma classe de câmera que comporte algumas formas de projeção do espaço-objeto para espaço-imagem

In [3]:
"""
Taken form https://github.com/graphdeco-inria/gaussian-splatting/blob/main/utils/graphics_utils.py
"""

import torch
import torch.nn as nn
import math
import os
import imageio.v2 as imageio

def getWorld2View2(R, t, translate=torch.tensor([.0, .0, .0]), scale=1.0):
    Rt = torch.eye(4).type(torch.float64)
    Rt[:3, :3] = R
    Rt[:3, 3] = t

    C2W = torch.linalg.inv(Rt)
    cam_center = -R.T @ t
    cam_center = (cam_center + translate) * scale
    C2W[:3, 3] = cam_center
    Rt = torch.linalg.inv(C2W)
    return Rt.cuda(), cam_center

def getProjectionMatrix(znear, zfar, fovX, fovY):
    """
    The function constructs a perspective projection matrix that maps 3D 
    camera/view coordinates into normalized device coordinates (NDC), 
    which range from -1 to 1.
    """
    tanHalfFovY = math.tan((fovY / 2))
    tanHalfFovX = math.tan((fovX / 2))

    top = tanHalfFovY * znear
    bottom = -top
    right = tanHalfFovX * znear
    left = -right

    P = torch.zeros(4, 4, dtype=torch.float64)

    z_sign = 1.0

    P[0, 0] = 2.0 * znear / (right - left)
    P[1, 1] = 2.0 * znear / (top - bottom)
    P[0, 2] = (right + left) / (right - left)
    P[1, 2] = (top + bottom) / (top - bottom)
    P[3, 2] = z_sign
    P[2, 2] = z_sign * zfar / (zfar - znear)
    P[2, 3] = -(zfar * znear) / (zfar - znear)
    return P

def fov2focal(fov, pixels):
    return pixels / (2 * math.tan(fov / 2))

def focal2fov(focal, pixels):
    return 2*math.atan(pixels/(2*focal))
    
class ExtCamera(nn.Module):
    def __init__(self, 
                 camera, 
                 image,
                 images_folder
                 ):
        super(ExtCamera, self).__init__()

        self.image_id = image.id
        self.camera_id = camera.id

        # Extract from intrinsics
        self.K = torch.from_numpy(intr_from_pinhole_camera(camera)).type(torch.float64).cuda()
        self.height = camera.height
        self.width = camera.width
        if camera.model=="SIMPLE_PINHOLE":
            focal_length_x = camera.params[0]
            FovY = focal2fov(focal_length_x, self.height)
            FovX = focal2fov(focal_length_x, self.width)
        elif camera.model=="PINHOLE":
            focal_length_x = camera.params[0]
            focal_length_y = camera.params[1]
            FovY = focal2fov(focal_length_y, self.height)
            FovX = focal2fov(focal_length_x, self.width)
        else:
            assert False, "Colmap camera model not handled: only undistorted datasets (PINHOLE or SIMPLE_PINHOLE cameras) supported!"
        self.FoVx = FovX
        self.FoVy = FovY    

        # Extract from extrinsics
        self.valid_mask = (image.point3D_ids != -1)
        self.point3D_ids = image.point3D_ids[self.valid_mask]
        self.R = torch.from_numpy(image.qvec2rotmat())
        self.t = torch.from_numpy(image.tvec)
        self.T , self.camera_center = getWorld2View2(self.R, self.t)
        self.image_name = image.name
        image_path = os.path.join(images_folder, image.name)
        self.gtimage = imageio.imread(image_path).astype(np.float32)  / 255.0        

        # Pinhole Projection:
        # Ponto 3D no mundo  --> (view matrix) -->
        # camera space --> (projection matrix) -->
        # screen space (tela/imagem)
        P = self.K @ self.T[:3, :]
        self.P = P

        # view matrix e projection matrix:
        # Se estiver renderizando em um motor gráfico (Unity, Unreal, OpenGL), 
        # vai usar view matrix e projection matrix:
        # Ponto 3D no mundo  --> (view matrix) -->
        # camera space --> (projection matrix) --> 
        # clip space --> (perspective divide) (dividir pela coord w)-->
        # Normalized Device Coordinates (NDC) --> (viewport) -->
        # screen space (tela/imagem)
        self.zfar = 100.0
        self.znear = 0.01
        # Matrizes em column-major: v' = v @ M.T
        self.projection_matrix = getProjectionMatrix(znear=self.znear, zfar=self.zfar, fovX=self.FoVx, fovY=self.FoVy).transpose(0,1).cuda() # K
        self.world_view_transform = self.T.transpose(0, 1).cuda() # T or w2camera or w2view

    def __repr__(self):
        return f'Camera({self.image_id:03}) with {len(self.point3D_ids):05} points 3D associated'
    


def readColmapCameras(cam_extrinsics, cam_intrinsics, images_folder):
    cam_infos = []
    for k, v in cam_extrinsics.items():
        cam_info = ExtCamera(camera = cam_intrinsics[v.camera_id],
                             image = v,
                             images_folder = images_folder)
        cam_infos.append(cam_info)
    return cam_infos

images_folder = os.path.join(folder,"images")
cameras = readColmapCameras(cam_extrinsics, cam_intrinsics, images_folder)
cameras

[Camera(001) with 00221 points 3D associated,
 Camera(002) with 00138 points 3D associated,
 Camera(003) with 00187 points 3D associated,
 Camera(004) with 00273 points 3D associated,
 Camera(005) with 00158 points 3D associated,
 Camera(006) with 00084 points 3D associated,
 Camera(007) with 00152 points 3D associated,
 Camera(008) with 00131 points 3D associated,
 Camera(009) with 00149 points 3D associated,
 Camera(010) with 00117 points 3D associated,
 Camera(011) with 00103 points 3D associated,
 Camera(012) with 00097 points 3D associated,
 Camera(013) with 00062 points 3D associated,
 Camera(024) with 00051 points 3D associated,
 Camera(025) with 00105 points 3D associated,
 Camera(026) with 00115 points 3D associated,
 Camera(027) with 00211 points 3D associated,
 Camera(028) with 00221 points 3D associated,
 Camera(029) with 00163 points 3D associated,
 Camera(030) with 00090 points 3D associated,
 Camera(031) with 00083 points 3D associated,
 Camera(032) with 00100 points 3D 

In [4]:
import time

def homogeneous(points):
    """
    homogeneous points
    :param points: [..., 3]
    """
    return torch.cat([points, torch.ones_like(points[..., :1])], dim=-1)

## 3 types of projection:
def projection_ndc(points, viewmatrix_T, projmatrix_T):
    """
    This function projects points from object space to screen space using a 
    view and projection matrix.
    It returns the projected points in NDC space and in view space.
    points: [..., 3] (object space)
    viewmatrix: (4, 4)
    projmatrix: (4, 4)
    """
    points_o = homogeneous(points) # object space  (N,4)
    points_h = points_o @ viewmatrix_T @ projmatrix_T # screen space # RHS
    p_w = 1.0 / (points_h[..., -1:] + 0.000001)
    p_proj = points_h * p_w
    p_view = points_o @ viewmatrix_T
    # in_mask = p_view[..., 2] >= 0.2
    return p_proj, p_view

def myprojection_ndc(points, viewmatrix, projmatrix):
    """
    This function projects points from object space to screen space using a 
    view and projection matrix.
    It returns the projected points in NDC space and in view space.
    points: [..., 3] (object space)
    viewmatrix: (4, 4)
    projmatrix: (4, 4)
    """
    points_o = homogeneous(points)     # object space (N,4)
    p_view   = viewmatrix @ points_o.T # view space   (4,N)
    points_h = projmatrix @ p_view     # NDC space (4,N) homogeneous
    p_proj = points_h / (points_h[-1, ...] + 0.000001) # NDC space (4,N)
    # in_mask = p_view[..., 2] >= 0.2
    return p_proj.T, p_view.T

def projection_pinhole(points, T, K):
    """
    This function projects points from object space to screen space using
    extrinsic and intrinsic matrices.
    points: [..., 3] (object space)
    intrinsicmatrix (K): (3, 3)
    extrinsicmatrix (T): (3, 4)
    """
    points_o = homogeneous(points) # object space (N,4)
    p_view = T @ points_o.T # view space (4,N)
    points_h = K @ p_view[:3, :] # screen space (3,N) homogeneous
    p_proj = points_h[:2] / (points_h[2, ...] + 0.000001) # screen space (3,N)
    return p_proj.T, p_view.T


### Time analisys with 3 types of projection:

In [6]:

print("Experiment with intrinsic and extrinsic matrices:")
start_time = time.time()
error_tot = torch.tensor([0.0,0.0]).cuda()
for camera in cameras:    
    ## Get data
    image = cam_extrinsics[camera.image_id]
    uv_colmap = torch.from_numpy(image.xys[camera.valid_mask]).cuda()

    points3d_filtred = [points3d[id] for id in camera.point3D_ids]
    points3d_xyz_filtred = torch.from_numpy(np.array([p.xyz for p in points3d_filtred])).type(torch.float64).cuda()

    ## Old proj method
    uv_calc, p_view = projection_pinhole(points3d_xyz_filtred, camera.T, camera.K)

    # Erro total
    error = torch.linalg.norm(uv_calc - uv_colmap, axis=0) # axis=0 is over columns, axis=1 is over rows
    error_tot += error
print(f"Total error:", error_tot.tolist())
print(f"Elapsed time: {time.time() - start_time:.6f} seconds\n")

# Delete references to unused objects
del image
del uv_colmap
del points3d_filtred
del points3d_xyz_filtred
del uv_calc
del p_view
del error
del error_tot
# Clear PyTorch's CUDA cache
torch.cuda.empty_cache()

print("Experiment with view matrix and projection matrix transposed "\
      "(col-major order), used in vanilla 3DGS and torch-splatting:" )
start_time = time.time()
error_tot = torch.tensor([0.0,0.0]).cuda()
for camera in cameras:    
    ## Get data
    image = cam_extrinsics[camera.image_id]
    uv_colmap = torch.from_numpy(image.xys[camera.valid_mask]).cuda()

    points3d_filtred = [points3d[id] for id in camera.point3D_ids]
    points3d_xyz_filtred = torch.tensor([p.xyz for p in points3d_filtred],dtype=torch.float64).cuda()

    ## New proj method
    p_proj, p_view = projection_ndc(points3d_xyz_filtred, camera.world_view_transform, camera.projection_matrix)
    mean_coord_x = ((p_proj[..., 0] + 1) * camera.width - 1.0) * 0.5
    mean_coord_y = ((p_proj[..., 1] + 1) * camera.height - 1.0) * 0.5
    uv_calc = torch.stack([mean_coord_x, mean_coord_y], dim=-1)
    depths = p_view[:,2]

    # Erro total
    error = torch.linalg.norm(uv_calc - uv_colmap, axis=0) # axis=0 is over columns, axis=1 is over rows
    error_tot += error
print(f"Total error:", error_tot.tolist())
print(f"Elapsed time: {time.time() - start_time:.6f} seconds\n")

# Delete references to unused objects
del image
del uv_colmap
del points3d_filtred
del points3d_xyz_filtred
del uv_calc
del p_view
del p_proj
del depths
del error
del error_tot
del mean_coord_x
del mean_coord_y
# Clear PyTorch's CUDA cache
torch.cuda.empty_cache()


print("Experiment with view matrix and projection matrix not transposed " \
      "(back to row-major order)")
start_time = time.time()
error_tot = torch.tensor([0.0,0.0]).cuda()
for camera in cameras:    
    ## Get data
    image = cam_extrinsics[camera.image_id]
    uv_colmap = torch.from_numpy(image.xys[camera.valid_mask]).cuda()

    points3d_filtred = [points3d[id] for id in camera.point3D_ids]
    points3d_xyz_filtred = torch.tensor([p.xyz for p in points3d_filtred],dtype=torch.float64).cuda()

    ## New proj method modified
    p_proj, p_view = myprojection_ndc(points3d_xyz_filtred, camera.world_view_transform.T, camera.projection_matrix.T)
    mean_coord_x = ((p_proj[..., 0] + 1) * camera.width - 1.0) * 0.5
    mean_coord_y = ((p_proj[..., 1] + 1) * camera.height - 1.0) * 0.5
    uv_calc = torch.stack([mean_coord_x, mean_coord_y], dim=-1)
    depths = p_view[:,2]
    
    # Erro total
    error = torch.linalg.norm(uv_calc - uv_colmap, axis=0) # axis=0 is over columns, axis=1 is over rows
    error_tot += error
print(f"Total error:", error_tot.tolist())
print(f"Elapsed time: {time.time() - start_time:.6f} seconds")

# Delete references to unused objects
del image
del uv_colmap
del points3d_filtred
del points3d_xyz_filtred
del uv_calc
del p_view
del p_proj
del depths
del error
del error_tot
del mean_coord_x
del mean_coord_y

# Clear PyTorch's CUDA cache
torch.cuda.empty_cache()

Experiment with intrinsic and extrinsic matrices:
Total error: [212.38931274414062, 158.12753295898438]
Elapsed time: 0.021773 seconds

Experiment with view matrix and projection matrix transposed (col-major order), used in vanilla 3DGS and torch-splatting:
Total error: [262.3323059082031, 220.69557189941406]
Elapsed time: 0.019452 seconds

Experiment with view matrix and projection matrix not transposed (back to row-major order)
Total error: [262.3323059082031, 220.69557189941406]
Elapsed time: 0.017004 seconds
